In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
model = joblib.load("/content/drive/MyDrive/data_energy/backup_models_campuswatt/rfr_model.pkl")

In [6]:
print(model.feature_names_in_)

['building_id' 'meter' 'site_id' 'air_temperature' 'cloud_coverage'
 'dew_temperature' 'precip_depth_1_hr' 'sea_level_pressure'
 'wind_direction' 'wind_speed' 'cooling_degree' 'heating_degree'
 'temp_dew_gap' 'humidity_proxy' 'feels_like_temp' 'is_raining'
 'weather_stress' 'storm_intensity' 'wind_x' 'wind_y' 'wind_energy'
 'pressure_anomaly' 'sunlight_proxy' 'thermal_load_proxy'
 'environmental_load_index']


In [3]:
def feature_engineer_energy(df, train_stats=None, is_train=True):
    df = df.copy()

    df = df.sort_values(["building_id", "meter", "timestamp"])

    group_cols = ["building_id", "meter"]


    temp = df["air_temperature"]

    df["cooling_degree"] = np.maximum(temp - 18, 0)
    df["heating_degree"] = np.maximum(18 - temp, 0)

    df["temp_dew_gap"] = df["air_temperature"] - df["dew_temperature"]

    df["humidity_proxy"] = df["dew_temperature"] / (df["air_temperature"] + 1e-6)

    df["feels_like_temp"] = df["air_temperature"] - 0.7 * df["wind_speed"]


    df["is_raining"] = (df["precip_depth_1_hr"] > 0).astype(int)

    df["weather_stress"] = (
        df["wind_speed"] +
        df["cloud_coverage"] +
        df["is_raining"]
    )

    df["storm_intensity"] = (
        df["wind_speed"] ** 2 +
        df["cloud_coverage"]
    )

    df["wind_x"] = np.sin(np.deg2rad(df["wind_direction"]))
    df["wind_y"] = np.cos(np.deg2rad(df["wind_direction"]))
    df["wind_energy"] = df["wind_speed"] ** 2

    if train_stats is not None:
        pressure_mean = train_stats["pressure_mean"]
    else:
        pressure_mean = df["sea_level_pressure"].mean()

    df["pressure_anomaly"] = df["sea_level_pressure"] - pressure_mean


    df["sunlight_proxy"] = 8 - df["cloud_coverage"]


    df["thermal_load_proxy"] = (
        df["cooling_degree"] + df["heating_degree"]
    )

    df["environmental_load_index"] = (
        df["weather_stress"] + np.abs(df["pressure_anomaly"])
    )


    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)

    return df

In [ ]:
test = pd.read_csv("/content/drive/MyDrive/data_energy/test_data_preprocessed.csv")
test = test.set_index('timestamp')
ids = test["row_id"]
test = test.drop(columns=["row_id"], axis=1)
print(test.columns.tolist())
test = feature_engineer_energy(test, is_train=False)


preds = model.predict(test)

submission = pd.DataFrame({
    "row_id": ids,
    "meter_reading": np.expm1(preds)
})

submission.to_csv("Ashrae_submission.csv", index=False)

print("Submission Completed!")

['building_id', 'meter', 'site_id', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']
